# iLQR Swing-Up for a Damped Cart-Pole

The cart-pole swing-up is a small example with the right difficulty: the dynamics are nonlinear, the target is unstable, and an open-loop guess is not enough.

The plan is to build an iLQR solver around the damped cart-pole model, first with a fixed terminal matrix, then with a terminal value matrix \(S_\infty\) computed from local infinite-horizon LQR at the upright equilibrium. The last section visualizes both optimized trajectories.

In [1]:
# Numerical arrays for simulation and symbolic algebra for automatic derivatives.
import numpy as np
import sympy as sp
from dataclasses import dataclass

## Problem setup

Use the state and input

$$
\boldsymbol{x}
=
\begin{bmatrix}
p & \theta & \dot p & \dot\theta
\end{bmatrix}^T,
\qquad
\boldsymbol{u}
=
\begin{bmatrix}
F
\end{bmatrix}.
$$

The initial condition is the downward equilibrium,

$$
\boldsymbol{x}_0=
\begin{bmatrix}
0 & 0 & 0 & 0
\end{bmatrix}^T,
$$

and the desired fixed point is the upright equilibrium,

$$
\boldsymbol{x}_d=
\begin{bmatrix}
0 & \pi & 0 & 0
\end{bmatrix}^T.
$$

The model includes both cart damping and pole damping:

$$
m_c=10,\quad
m_p=2,\quad
l=0.5,\quad
g=9.8,\quad
b=0.1,\quad
d=0.1.
$$

The horizon is short enough to force a real swing-up, but long enough for the optimizer to discover a smooth maneuver.

In [2]:
# Dimensions.
n_x = 4
n_u = 1

# Horizon and discretization.
T = 2.5
dt = 0.01
N = int(T / dt)        # Number of state samples, matching the MATLAB convention.
N_u = N - 1            # Number of control intervals.

# Boundary states.
x0 = np.array([0.0, 0.0, 0.0, 0.0])
xd = np.array([0.0, np.pi, 0.0, 0.0])

# Moderate target-centered swing-up weights.
# Qf is used only in the first run; the second run replaces it with S_inf.
Q = np.diag([2.0, 2.0, 0.1, 0.1])
Qf = np.diag([200.0, 2000.0, 100.0, 300.0])
R = np.array([[0.003]])

## Damped cart-pole dynamics

Let

$$
D(\theta)=m_c+m_p\sin^2\theta.
$$

The first two equations only define the velocities:

$$
\dot p=\dot p,
\qquad
\dot\theta=\dot\theta.
$$

The acceleration equations are the model:

$$
\ddot p
=
\frac{
F-b\dot p+\frac{d}{l}\dot\theta\cos\theta
+
m_p\sin\theta\left(l\dot\theta^2+g\cos\theta\right)
}{
D(\theta)
},
$$

$$
\ddot\theta
=
\frac{
-F\cos\theta
+b\dot p\cos\theta
-\frac{d(m_c+m_p)}{m_p l}\dot\theta
-m_p l\dot\theta^2\cos\theta\sin\theta
-\frac{(m_c+m_p)g}{m_p l}\sin\theta
}{
lD(\theta)
}.
$$

The discrete dynamics are a forward Euler step:

$$
\boldsymbol{x}_{n+1}
=
\boldsymbol{f}_d(\boldsymbol{x}_n,\boldsymbol{u}_n)
=
\boldsymbol{x}_n+h\boldsymbol{f}_c(\boldsymbol{x}_n,\boldsymbol{u}_n).
$$

The important implementation choice is to write the dynamics once and differentiate them automatically. This keeps the notebook close to the equations while avoiding a long hand-written Jacobian.

In [3]:
# Physical parameters for the damped cart-pole model.
@dataclass
class CartPoleParams:
    mc: float = 10.0
    mp: float = 2.0
    l: float = 0.5
    g: float = 9.8
    b: float = 0.1
    d: float = 0.1


class CartPoleModel:
    """Damped cart-pole model using the MATLAB dynamics."""

    def __init__(self, params: CartPoleParams, dt: float):
        self.params = params
        self.dt = dt

    @staticmethod
    def _is_symbolic(x, u):
        # The same dynamics function is used for NumPy rollouts and SymPy derivatives.
        return np.asarray(x).dtype == object or np.asarray(u).dtype == object

    def continuous_dynamics(self, x, u):
        symbolic = self._is_symbolic(x, u)
        x = np.asarray(x, dtype=object if symbolic else float)
        u = np.asarray(u, dtype=object if symbolic else float).reshape(-1)[0]
        m = sp if symbolic else np

        mc, mp, l, g, b, d = self.params.mc, self.params.mp, self.params.l, self.params.g, self.params.b, self.params.d
        _, theta, p_dot, theta_dot = x
        s, c = m.sin(theta), m.cos(theta)
        D = mc + mp * s**2

        # Four first-order state derivatives: [p_dot, theta_dot, p_ddot, theta_ddot].
        x_dot = np.array([
            p_dot,
            theta_dot,
            (u - b * p_dot + d * theta_dot * c / l + mp * s * (l * theta_dot**2 + g * c)) / D,
            (-u * c + b * p_dot * c - d * (mc + mp) * theta_dot / (mp * l) - mp * l * theta_dot**2 * c * s - (mc + mp) * g * s / (mp * l)) / (l * D),
        ], dtype=object if symbolic else float)

        return x_dot

    def discrete_dynamics(self, x, u):
        # Forward Euler discretization used by the optimizer.
        symbolic = self._is_symbolic(x, u)
        x = np.asarray(x, dtype=object if symbolic else float)
        return x + self.dt * self.continuous_dynamics(x, u)


model = CartPoleModel(CartPoleParams(), dt)

## Cost functions

A swing-up cost should reward the upright target, not the downward origin. Therefore the running cost is centered at \(\boldsymbol{x}_d\):

$$
\ell(\boldsymbol{x},\boldsymbol{u})
=
\frac{1}{2}
(\boldsymbol{x}-\boldsymbol{x}_d)^T
Q
(\boldsymbol{x}-\boldsymbol{x}_d)
+
\frac{1}{2}
\boldsymbol{u}^T R \boldsymbol{u}.
$$

The first run uses a fixed terminal matrix,

$$
\ell_f(\boldsymbol{x})
=
\frac{1}{2}
(\boldsymbol{x}-\boldsymbol{x}_d)^T
Q_f
(\boldsymbol{x}-\boldsymbol{x}_d).
$$

With this convention,

$$
\ell_{\boldsymbol{x}}=Q(\boldsymbol{x}-\boldsymbol{x}_d),
\qquad
\ell_{\boldsymbol{u}}=R\boldsymbol{u},
\qquad
\ell_{\boldsymbol{xx}}=Q,
\qquad
\ell_{\boldsymbol{ux}}=0,
\qquad
\ell_{\boldsymbol{uu}}=R,
$$

and

$$
\ell_{f,\boldsymbol{x}}=Q_f(\boldsymbol{x}-\boldsymbol{x}_d),
\qquad
\ell_{f,\boldsymbol{xx}}=Q_f.
$$

The target-centered stage cost is the small detail that makes moderate weights usable. If the running cost were centered at the origin, the optimizer would be asked to swing up while also being punished for going near \(\theta=\pi\).

In [4]:
def cost_stage(x, u):
    # Target-centered quadratic cost for the swing-up task.
    x = np.asarray(x)
    u = np.asarray(u).reshape(n_u)
    e = x - xd
    return 0.5 * e @ Q @ e + 0.5 * u @ R @ u


def cost_final(x):
    # Fixed terminal matrix used in the first solve.
    x = np.asarray(x)
    e = x - xd
    return 0.5 * e @ Qf @ e


class CartPoleCost:
    """Target-centered quadratic running cost and terminal cost."""

    def stage(self, x, u):
        return float(cost_stage(np.asarray(x, dtype=float), np.asarray(u, dtype=float)))

    def final(self, x):
        return float(cost_final(np.asarray(x, dtype=float)))


cost = CartPoleCost()


def rollout(model, x0, u_trj):
    # Roll out the nonlinear dynamics under an open-loop control sequence.
    x_trj = np.zeros((u_trj.shape[0] + 1, n_x))
    x_trj[0, :] = x0

    for n in range(u_trj.shape[0]):
        x_trj[n + 1, :] = model.discrete_dynamics(x_trj[n, :], u_trj[n, :])

    return x_trj


def cost_trj(cost, x_trj, u_trj):
    # True nonlinear trajectory cost used to accept or reject iLQR steps.
    total = 0.0

    for n in range(u_trj.shape[0]):
        total += cost.stage(x_trj[n, :], u_trj[n, :])

    total += cost.final(x_trj[-1, :])
    return float(total)


## Symbolic derivatives

iLQR needs local linear-quadratic models along the current trajectory. At each time step, we need

$$
\frac{\partial \boldsymbol{f}_d}{\partial \boldsymbol{x}},
\qquad
\frac{\partial \boldsymbol{f}_d}{\partial \boldsymbol{u}},
\qquad
\ell_{\boldsymbol{x}},\ell_{\boldsymbol{u}},
\ell_{\boldsymbol{xx}},\ell_{\boldsymbol{ux}},\ell_{\boldsymbol{uu}}.
$$

The code below builds these derivatives symbolically once, then evaluates the resulting functions numerically during the backward pass. Conceptually, the solver sees a fresh local LQR problem around the current nominal trajectory at every iteration.

In [5]:
class Derivatives:
    def __init__(self, discrete_dynamics, cost_stage, cost_final, n_x, n_u):
        # Symbolic variables for one time step.
        self.x_sym = np.array(sp.symbols(f"x_0:{n_x}"), dtype=object)
        self.u_sym = np.array(sp.symbols(f"u_0:{n_u}"), dtype=object)

        x_vec = sp.Matrix(self.x_sym)
        u_vec = sp.Matrix(self.u_sym)
        args = tuple(self.x_sym) + tuple(self.u_sym)

        # Stage-cost first and second derivatives.
        l = sp.sympify(cost_stage(self.x_sym, self.u_sym))
        l_x = sp.Matrix([sp.diff(l, x_i) for x_i in self.x_sym])
        l_u = sp.Matrix([sp.diff(l, u_i) for u_i in self.u_sym])
        l_xx = l_x.jacobian(x_vec)
        l_ux = l_u.jacobian(x_vec)
        l_uu = l_u.jacobian(u_vec)

        # Terminal-cost derivatives.
        l_final = sp.sympify(cost_final(self.x_sym))
        l_final_x = sp.Matrix([sp.diff(l_final, x_i) for x_i in self.x_sym])
        l_final_xx = l_final_x.jacobian(x_vec)

        # Discrete dynamics Jacobians.
        f = sp.Matrix(discrete_dynamics(self.x_sym, self.u_sym))
        f_x = f.jacobian(x_vec)
        f_u = f.jacobian(u_vec)

        # Compile symbolic expressions into fast numerical functions.
        self._stage_fn = sp.lambdify(args, [l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u], modules="numpy")
        self._final_fn = sp.lambdify(tuple(self.x_sym), [l_final_x, l_final_xx], modules="numpy")

    def stage(self, x, u):
        # Evaluate all local derivatives needed by one backward-pass step.
        values = self._stage_fn(*np.asarray(x, dtype=float).reshape(n_x), *np.asarray(u, dtype=float).reshape(n_u))
        l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u = [np.asarray(v, dtype=float) for v in values]
        return l_x.ravel(), l_u.ravel(), l_xx, l_ux, l_uu, f_x, f_u

    def final(self, x):
        # Initialize the backward pass from the terminal value derivatives.
        values = self._final_fn(*np.asarray(x, dtype=float).reshape(n_x))
        l_final_x, l_final_xx = [np.asarray(v, dtype=float) for v in values]
        return l_final_x.ravel(), l_final_xx


derivs = Derivatives(model.discrete_dynamics, cost_stage, cost_final, n_x, n_u)


## One-step Bellman expansion

The finite-horizon problem is

$$
\begin{aligned}
\min_{\boldsymbol{u}_0,\ldots,\boldsymbol{u}_{N-1}}
\quad&
\ell_f(\boldsymbol{x}_N)
+
\sum_{n=0}^{N-1}
\ell(\boldsymbol{x}_n,\boldsymbol{u}_n)
\\
\text{subject to}\quad&
\boldsymbol{x}_{n+1}
=
\boldsymbol{f}_d(\boldsymbol{x}_n,\boldsymbol{u}_n),
\qquad
\boldsymbol{x}_0\ \text{given}.
\end{aligned}
$$

iLQR applies dynamic programming locally. Define

$$
Q_n(\boldsymbol{x},\boldsymbol{u})
=
\ell(\boldsymbol{x},\boldsymbol{u})
+
V_{n+1}(\boldsymbol{f}_d(\boldsymbol{x},\boldsymbol{u})).
$$

Around a nominal pair \((\bar{\boldsymbol{x}}_n,\bar{\boldsymbol{u}}_n)\), write

$$
\delta\boldsymbol{x}
=
\boldsymbol{x}-\bar{\boldsymbol{x}}_n,
\qquad
\delta\boldsymbol{u}
=
\boldsymbol{u}-\bar{\boldsymbol{u}}_n.
$$

Using

$$
\delta\boldsymbol{x}_{n+1}
\approx
\boldsymbol{f}_{\boldsymbol{x},n}\delta\boldsymbol{x}
+
\boldsymbol{f}_{\boldsymbol{u},n}\delta\boldsymbol{u},
$$

the quadratic coefficients are

$$
\begin{aligned}
Q_{\boldsymbol{x}}
&=
\ell_{\boldsymbol{x}}
+
\boldsymbol{f}_{\boldsymbol{x}}^T V_{\boldsymbol{x}},
&
Q_{\boldsymbol{u}}
&=
\ell_{\boldsymbol{u}}
+
\boldsymbol{f}_{\boldsymbol{u}}^T V_{\boldsymbol{x}},
\\
Q_{\boldsymbol{xx}}
&=
\ell_{\boldsymbol{xx}}
+
\boldsymbol{f}_{\boldsymbol{x}}^T V_{\boldsymbol{xx}}\boldsymbol{f}_{\boldsymbol{x}},
&
Q_{\boldsymbol{ux}}
&=
\ell_{\boldsymbol{ux}}
+
\boldsymbol{f}_{\boldsymbol{u}}^T V_{\boldsymbol{xx}}\boldsymbol{f}_{\boldsymbol{x}},
\\
Q_{\boldsymbol{uu}}
&=
\ell_{\boldsymbol{uu}}
+
\boldsymbol{f}_{\boldsymbol{u}}^T V_{\boldsymbol{xx}}\boldsymbol{f}_{\boldsymbol{u}}.
\end{aligned}
$$

In [6]:
def Q_terms(l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx):
    # Local quadratic model of ell(x, u) + V(f(x, u)).
    Q_x = l_x + f_x.T @ V_x
    Q_u = l_u + f_u.T @ V_x
    Q_xx = l_xx + f_x.T @ V_xx @ f_x
    Q_ux = l_ux + f_u.T @ V_xx @ f_x
    Q_uu = l_uu + f_u.T @ V_xx @ f_u

    return Q_x, Q_u, Q_xx, Q_ux, Q_uu

## Feedback law and value update

The local quadratic model is minimized over \(\delta\boldsymbol{u}\) while holding \(\delta\boldsymbol{x}\) fixed:

$$
Q_{\boldsymbol{u}}
+
Q_{\boldsymbol{ux}}\delta\boldsymbol{x}
+
Q_{\boldsymbol{uu}}\delta\boldsymbol{u}
=
0.
$$

Therefore the control update has the form

$$
\delta\boldsymbol{u}^*
=
\boldsymbol{k}
+
K\delta\boldsymbol{x},
$$

with

$$
\boldsymbol{k}
=
-Q_{\boldsymbol{uu}}^{-1}Q_{\boldsymbol{u}},
\qquad
K
=
-Q_{\boldsymbol{uu}}^{-1}Q_{\boldsymbol{ux}}.
$$

The feedforward term \(\boldsymbol{k}\) moves the nominal control sequence. The feedback term \(K\delta\boldsymbol{x}\) keeps the new rollout close to the trajectory used to compute the derivatives.

After substituting this optimal local policy back into the quadratic model, the value derivatives become

$$
V_{\boldsymbol{x}}
=
Q_{\boldsymbol{x}}
+
K^TQ_{\boldsymbol{u}}
+
Q_{\boldsymbol{ux}}^T\boldsymbol{k}
+
K^TQ_{\boldsymbol{uu}}\boldsymbol{k},
$$

$$
V_{\boldsymbol{xx}}
=
Q_{\boldsymbol{xx}}
+
K^TQ_{\boldsymbol{ux}}
+
Q_{\boldsymbol{ux}}^TK
+
K^TQ_{\boldsymbol{uu}}K.
$$

This is the Riccati recursion written for a nonlinear trajectory.

In [7]:
def gains(Q_uu, Q_u, Q_ux):
    # Minimize the local quadratic Q-function with respect to delta-u.
    k = -np.linalg.solve(Q_uu, Q_u)
    K = -np.linalg.solve(Q_uu, Q_ux)
    return k, K


def V_terms(Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k):
    # Push the optimized local Q-function backward into the value function.
    V_x = Q_x + K.T @ Q_u + Q_ux.T @ k + K.T @ Q_uu @ k
    V_xx = Q_xx + K.T @ Q_ux + Q_ux.T @ K + K.T @ Q_uu @ K

    # Symmetrize for numerical hygiene.
    V_xx = 0.5 * (V_xx + V_xx.T)

    return V_x, V_xx


def expected_cost_reduction(Q_u, Q_uu, k):
    # First-order diagnostic for how much the local model expects to improve.
    return float(-Q_u.T @ k - 0.5 * k.T @ Q_uu @ k)

## Backward pass, forward pass, and line search

Each iLQR iteration has two parts.

The backward pass computes \(\boldsymbol{k}_n\) and \(K_n\) from the final time back to the initial time. The forward pass then rolls out the full nonlinear dynamics using

$$
\boldsymbol{u}'_n
=
\bar{\boldsymbol{u}}_n
+
\alpha\boldsymbol{k}_n
+
K_n(\boldsymbol{x}'_n-\bar{\boldsymbol{x}}_n).
$$

The scalar \(\alpha\) is a line-search parameter. We first try the full step, then smaller steps.

The main numerical safeguard is to regularize the control Hessian:

$$
Q_{\boldsymbol{uu}}^{\mathrm{reg}}
=
Q_{\boldsymbol{uu}}+\rho I.
$$

If the local model is too aggressive, increasing \(\rho\) makes the step more conservative. If the rollout decreases the cost, \(\rho\) is relaxed. This is the difference between the clean Riccati equation on paper and a robust iLQR loop in code.

In [8]:
class ILQRSolver:
    def __init__(
        self,
        model,
        cost,
        derivs,
        max_iter=300,
        regu_init=1.0,
        min_regu=1e-9,
        max_regu=1e10,
        alphas=(1.0, 0.5, 0.25, 0.1, 0.05, 0.025, 0.01, 0.005, 0.001),
        tol=1e-7,
        u_clip=None,
    ):
        self.model = model
        self.cost = cost
        self.derivs = derivs
        self.max_iter = max_iter
        self.regu_init = regu_init
        self.min_regu = min_regu
        self.max_regu = max_regu
        self.alphas = alphas
        self.tol = tol
        self.u_clip = u_clip

    def backward_pass(self, x_trj, u_trj, regu):
        # Compute local feedback laws from the end of the trajectory backward.
        k_trj = np.zeros((u_trj.shape[0], n_u))
        K_trj = np.zeros((u_trj.shape[0], n_u, n_x))
        expected_reduction = 0.0

        # Terminal value initializes the Riccati recursion.
        V_x, V_xx = self.derivs.final(x_trj[-1, :])

        for n in range(u_trj.shape[0] - 1, -1, -1):
            l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u = self.derivs.stage(
                x_trj[n, :], u_trj[n, :]
            )

            Q_x, Q_u, Q_xx, Q_ux, Q_uu = Q_terms(
                l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx
            )

            # Regularize only the control Hessian solve.
            Q_uu_reg = Q_uu + regu * np.eye(n_u)
            k, K = gains(Q_uu_reg, Q_u, Q_ux)

            k_trj[n, :] = k
            K_trj[n, :, :] = K

            # Match the driving notebook: gains use regularized Q_uu,
            # while the value update keeps the original quadratic model.
            V_x, V_xx = V_terms(Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k)
            expected_reduction += expected_cost_reduction(Q_u, Q_uu, k)

            if not (np.all(np.isfinite(V_x)) and np.all(np.isfinite(V_xx))):
                raise FloatingPointError("Backward pass produced non-finite value derivatives.")

        return k_trj, K_trj, expected_reduction

    def forward_pass(self, x_trj, u_trj, k_trj, K_trj, alpha):
        # Roll out the nonlinear dynamics under the candidate iLQR update.
        x_new = np.zeros_like(x_trj)
        u_new = np.zeros_like(u_trj)
        x_new[0, :] = x_trj[0, :]

        for n in range(u_trj.shape[0]):
            # Feedback is evaluated on the deviation from the old nominal trajectory.
            dx = x_new[n, :] - x_trj[n, :]
            u_new[n, :] = u_trj[n, :] + alpha * k_trj[n, :] + K_trj[n, :, :] @ dx

            if self.u_clip is not None:
                u_new[n, :] = np.clip(u_new[n, :], -self.u_clip, self.u_clip)

            x_new[n + 1, :] = self.model.discrete_dynamics(x_new[n, :], u_new[n, :])

            if not (np.all(np.isfinite(x_new[n + 1, :])) and np.all(np.isfinite(u_new[n, :]))):
                raise FloatingPointError("Forward rollout produced non-finite values.")

        return x_new, u_new

    def solve(self, x0, N_u, u_init=None):
        # Start from zero controls unless an initial guess is provided.
        if u_init is None:
            u_trj = np.zeros((N_u, n_u))
        else:
            u_trj = np.asarray(u_init, dtype=float).reshape(N_u, n_u)

        x_trj = rollout(self.model, x0, u_trj)
        J = cost_trj(self.cost, x_trj, u_trj)
        regu = self.regu_init

        trace = {
            "cost": [J],
            "regu": [regu],
            "alpha": [],
            "expected_reduction": [],
            "actual_reduction": [],
            "accepted": [],
        }

        for it in range(self.max_iter):
            try:
                k_trj, K_trj, expected_reduction = self.backward_pass(x_trj, u_trj, regu)
            except (np.linalg.LinAlgError, FloatingPointError):
                regu = min(self.max_regu, 10.0 * regu)
                trace["accepted"].append(False)
                continue

            # Try a sequence of step sizes until the true nonlinear cost decreases.
            accepted = False

            for alpha in self.alphas:
                try:
                    x_candidate, u_candidate = self.forward_pass(
                        x_trj, u_trj, k_trj, K_trj, alpha
                    )
                    J_candidate = cost_trj(self.cost, x_candidate, u_candidate)
                except FloatingPointError:
                    continue

                actual_reduction = J - J_candidate

                if np.isfinite(J_candidate) and actual_reduction > 0.0:
                    x_trj = x_candidate
                    u_trj = u_candidate
                    J = J_candidate
                    regu = max(self.min_regu, 0.5 * regu)

                    trace["cost"].append(J)
                    trace["regu"].append(regu)
                    trace["alpha"].append(alpha)
                    trace["expected_reduction"].append(expected_reduction)
                    trace["actual_reduction"].append(actual_reduction)
                    trace["accepted"].append(True)

                    accepted = True
                    break

            # If no rollout worked, trust the local model less next time.
            if not accepted:
                regu = min(self.max_regu, 10.0 * regu)
                trace["cost"].append(J)
                trace["regu"].append(regu)
                trace["alpha"].append(0.0)
                trace["expected_reduction"].append(expected_reduction)
                trace["actual_reduction"].append(0.0)
                trace["accepted"].append(False)

            if accepted and abs(trace["actual_reduction"][-1]) < self.tol:
                break

        return x_trj, u_trj, trace

## First run: fixed terminal matrix

The first solve uses the target-centered running cost and the fixed terminal matrix \(Q_f\). The optimizer starts from zero controls, repeatedly constructs local LQR problems, and accepts only nonlinear rollouts that reduce the true trajectory cost.

The result is stored as

- `x_trj`: state trajectory;
- `u_trj`: input trajectory;
- `trace`: cost, regularization, step-size, and acceptance history.

In [9]:
# Solve the swing-up problem with the fixed terminal matrix Qf.
solver = ILQRSolver(
    model=model,
    cost=cost,
    derivs=derivs,
    max_iter=250,
    regu_init=1.0,
    tol=1e-6,
)

x_trj, u_trj, trace = solver.solve(x0=x0, N_u=N_u)

# Basic numerical summary of the optimized trajectory.
print("Number of accepted steps:", sum(trace["accepted"]))
print("Number of total iterations:", len(trace["accepted"]))
print("Initial cost:", trace["cost"][0])
print("Final cost:", trace["cost"][-1])
print("Final state x[N-1]:", x_trj[-1])
print("Target state xd:", xd)
print("Final error x[N-1] - xd:", x_trj[-1] - xd)
print("First five controls:")
print(u_trj[:5].ravel())
print("Last five controls:")
print(u_trj[-5:].ravel())
print("Max |u|:", np.max(np.abs(u_trj)))


Number of accepted steps: 196
Number of total iterations: 196
Initial cost: 12327.135896960612
Final cost: 2163.8337455585483
Final state x[N-1]: [-1.03476792e-01  3.14177424e+00  6.75200288e-02 -1.84361770e-03]
Target state xd: [0.         3.14159265 0.         0.        ]
Final error x[N-1] - xd: [-0.10347679  0.00018159  0.06752003 -0.00184362]
First five controls:
[81.38716335 81.76352558 81.95353959 81.95583567 81.76858216]
Last five controls:
[-1.62241731 -1.68788866 -1.75294389 -1.81761743 -1.88194408]
Max |u|: 149.826583886831


## Replace the terminal guess by \(S_\infty\)

A fixed diagonal \(Q_f\) says only that terminal error is bad. It does not know the dynamics near the upright equilibrium.

A better terminal cost comes from the infinite-horizon LQR problem around

$$
\boldsymbol{x}^\star=
\begin{bmatrix}
0 & \pi & 0 & 0
\end{bmatrix}^T,
\qquad
\boldsymbol{u}^\star=0.
$$

Linearize the discrete dynamics there:

$$
\bar{\boldsymbol{x}}_{n+1}
\approx
A\bar{\boldsymbol{x}}_n
+
B\bar{\boldsymbol{u}}_n.
$$

Then solve the discrete algebraic Riccati equation

$$
S_\infty
=
Q_{\mathrm{lqr}}
+
A^TS_\infty A
-
A^TS_\infty B
\left(
R_{\mathrm{lqr}}+B^TS_\infty B
\right)^{-1}
B^TS_\infty A.
$$

The terminal cost becomes

$$
\ell_f(\boldsymbol{x}_N)
=
\frac{1}{2}
(\boldsymbol{x}_N-\boldsymbol{x}^\star)^T
S_\infty
(\boldsymbol{x}_N-\boldsymbol{x}^\star).
$$

Now the meaning is stronger: iLQR is responsible for the nonlinear swing-up, while \(S_\infty\) approximates the remaining optimal cost of stabilizing the upright equilibrium forever.

In [10]:
from scipy.linalg import solve_discrete_are

# Upright equilibrium for the damped cart-pole.
x_star = xd.copy()
u_star = np.zeros(n_u)

# Linearize the discrete-time dynamics x[n+1] = f_d(x[n], u[n]) at the upright equilibrium.
# The current Derivatives object already contains the symbolic dynamics Jacobians.
_, _, _, _, _, A_eq, B_eq = derivs.stage(x_star, u_star)

# Local stabilization weights: these describe the small-error upright LQR problem.
Q_lqr = np.diag([2.0, 20.0, 1.0, 5.0])
R_lqr = np.array([[0.003]])

# Steady-state Riccati solution: the terminal value Hessian for the second iLQR run.
S_inf = solve_discrete_are(A_eq, B_eq, Q_lqr, R_lqr)
S_inf = 0.5 * (S_inf + S_inf.T)  # remove tiny numerical asymmetry

print("A_eq:")
print(A_eq)
print("B_eq:")
print(B_eq)
print("diag(S_inf):", np.diag(S_inf))


A_eq:
[[ 1.000e+00  0.000e+00  1.000e-02  0.000e+00]
 [ 0.000e+00  1.000e+00  0.000e+00  1.000e-02]
 [ 0.000e+00  1.960e-02  9.999e-01 -2.000e-04]
 [ 0.000e+00  2.352e-01 -2.000e-04  9.976e-01]]
B_eq:
[[0.   ]
 [0.   ]
 [0.001]
 [0.002]]
diag(S_inf): [ 349.48843161 7439.44565994  356.78587699  282.29774601]


Run the same iLQR solver again. Nothing about the dynamics, stage cost, backward pass, forward pass, regularization, or line search changes. Only the terminal value changes from a hand-chosen \(Q_f\) to the local LQR value \(S_\infty\).

In [11]:
def cost_final_lqr_terminal(x):
    # Terminal value from the local infinite-horizon LQR problem.
    x = np.asarray(x)
    e = x - x_star
    return 0.5 * e @ S_inf @ e


class CartPoleCostWithLQRTerminal:
    """Same target-centered stage cost, but terminal value from local infinite-horizon LQR."""

    def stage(self, x, u):
        return float(cost_stage(np.asarray(x, dtype=float), np.asarray(u, dtype=float)))

    def final(self, x):
        return float(cost_final_lqr_terminal(np.asarray(x, dtype=float)))


cost_lqr_terminal = CartPoleCostWithLQRTerminal()

# Rebuild derivatives because the terminal cost has changed.
derivs_lqr_terminal = Derivatives(
    model.discrete_dynamics,
    cost_stage,
    cost_final_lqr_terminal,
    n_x,
    n_u,
)

# Same iLQR machinery; only the terminal value is different.
solver_lqr_terminal = ILQRSolver(
    model=model,
    cost=cost_lqr_terminal,
    derivs=derivs_lqr_terminal,
    max_iter=250,
    regu_init=1.0,
    tol=1e-6,
)

x_trj_lqr_terminal, u_trj_lqr_terminal, trace_lqr_terminal = solver_lqr_terminal.solve(
    x0=x0,
    N_u=N_u,
)

# Basic numerical summary of the S_inf-terminal trajectory.
print("Number of accepted steps:", sum(trace_lqr_terminal["accepted"]))
print("Number of total iterations:", len(trace_lqr_terminal["accepted"]))
print("Initial cost:", trace_lqr_terminal["cost"][0])
print("Final cost:", trace_lqr_terminal["cost"][-1])
print("Final state x[N-1]:", x_trj_lqr_terminal[-1])
print("Target state xd:", xd)
print("Final error x[N-1] - xd:", x_trj_lqr_terminal[-1] - xd)
print("First five controls:")
print(u_trj_lqr_terminal[:5].ravel())
print("Last five controls:")
print(u_trj_lqr_terminal[-5:].ravel())
print("Max |u|:", np.max(np.abs(u_trj_lqr_terminal)))


Number of accepted steps: 210
Number of total iterations: 210
Initial cost: 39169.724309366284
Final cost: 2163.29159840797
Final state x[N-1]: [-0.10757839  3.14196872  0.10491469  0.0284578 ]
Target state xd: [0.         3.14159265 0.         0.        ]
Final error x[N-1] - xd: [-0.10757839  0.00037606  0.10491469  0.0284578 ]
First five controls:
[81.38001721 81.75070211 81.93455347 81.93021609 81.7358725 ]
Last five controls:
[-0.79787103 -0.86261173 -0.92755452 -0.99277138 -1.05833663]
Max |u|: 149.9468657786298


## Visualize both optimized trajectories

The final step is only visualization. The trajectories have already been computed by the damped cart-pole dynamics above.

The drawing code uses a planar cart-pole visualizer and force-publishes each optimized state. It does not re-simulate a different cart-pole model. Therefore the two animations show the two trajectories produced by the same optimizer:

- fixed terminal matrix \(S=Q_f\);
- local infinite-horizon terminal value \(S_\infty\).

This makes the comparison clean: the model, horizon, stage cost, and iLQR loop are the same; only the terminal value changes.

In [ ]:
# Drake is used here only for visualization; the optimized states are precomputed.
from IPython.display import HTML, display
from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    Parser,
    PlanarSceneGraphVisualizer,
    Simulator,
)
from underactuated import ConfigureParser


def build_cartpole_visualizer_like_balancing(
    xlim=[-3.0, 3.0],
    ylim=[-1.35, 1.35],
):
    """Build the same cart-pole visualizer used in cartpole_balancing.ipynb.

    The only small visual change is a slightly larger y-window than the
    reference notebook, which makes the pole look a little shorter on screen.
    The optimized trajectory itself is unchanged.
    """
    # Start the block-diagram construction.
    builder = DiagramBuilder()

    # Instantiate the cart-pole and the scene graph.
    cartpole, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(cartpole)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
    cartpole.Finalize()

    # Add the planar visualizer and wire it to the scene graph.
    visualizer = builder.AddSystem(
        PlanarSceneGraphVisualizer(
            scene_graph,
            xlim=xlim,
            ylim=ylim,
            show=False,
        )
    )
    builder.Connect(scene_graph.get_query_output_port(), visualizer.get_input_port(0))

    # Finish building the block diagram.
    diagram = builder.Build()

    # Instantiate a simulator so the diagram has a mutable context.
    simulator = Simulator(diagram)

    return diagram, simulator, cartpole, visualizer


def animate_precomputed_trajectory_like_balancing(
    x_trj,
    title,
    dt,
    stride=5,
    xlim=[-3.0, 3.0],
    ylim=[-1.35, 1.35],
):
    """Animate a precomputed cart-pole trajectory with the balancing-notebook visualizer.

    This function intentionally does not call Drake's cart-pole dynamics.
    It only uses Drake's PlanarSceneGraphVisualizer to draw the states that
    were already computed by the iLQR solver using the MATLAB damped dynamics.
    """
    x_trj = np.asarray(x_trj, dtype=float)
    frame_ids = np.arange(0, x_trj.shape[0], stride)
    if frame_ids[-1] != x_trj.shape[0] - 1:
        frame_ids = np.r_[frame_ids, x_trj.shape[0] - 1]

    diagram, simulator, cartpole, visualizer = build_cartpole_visualizer_like_balancing(
        xlim=xlim,
        ylim=ylim,
    )

    # Use the diagram context, but do not simulate the cart-pole dynamics.
    context = simulator.get_mutable_context()
    plant_context = cartpole.GetMyMutableContextFromRoot(context)

    # Initialize the visualizer at the first optimized state.
    context.SetTime(0.0)
    cartpole.SetPositionsAndVelocities(plant_context, x_trj[0])

    # Start recording the optimized trajectory.
    visualizer.start_recording()
    simulator.Initialize()

    for frame_id in frame_ids:
        x = x_trj[frame_id]
        context.SetTime(frame_id * dt)
        cartpole.SetPositionsAndVelocities(plant_context, x)
        diagram.ForcedPublish(context)

    # Stop recording.
    visualizer.stop_recording()

    # Convert the recording into a notebook animation.
    ani = visualizer.get_recording_as_animation()

    # Display the animation below the cell.
    print(title)
    display(HTML(ani.to_jshtml()))

    # Reset the recorder for the next trajectory.
    visualizer.reset_recording()

    return ani


In [ ]:
# Draw both terminal-cost choices with the same visualizer settings.
# Visualize the trajectory from the fixed diagonal terminal matrix S = Qf.
ani_fixed_S = animate_precomputed_trajectory_like_balancing(
    x_trj=x_trj,
    title="iLQR swing-up with fixed diagonal terminal S = Qf",
    dt=dt,
    stride=5,
)

# Visualize the trajectory from the local infinite-horizon LQR terminal value S_inf.
ani_S_inf = animate_precomputed_trajectory_like_balancing(
    x_trj=x_trj_lqr_terminal,
    title="iLQR swing-up with local infinite-horizon LQR terminal S_inf",
    dt=dt,
    stride=5,
)
